{
 "cells": [
  {
   "cell_type": "markdown",
   "id": "c3561c56",
   "metadata": {},
   "source": [
    "# DEEP NEURAL NETWORKS - ASSIGNMENT 2: CNN FOR IMAGE CLASSIFICATION\n",
    "\n",
    "## Convolutional Neural Networks: Custom Implementation vs Transfer Learning"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "fb7be6ad",
   "metadata": {},
   "source": [
    "STUDENT INFORMATION (REQUIRED - DO NOT DELETE)\n",
    "\n",
    "BITS ID: 2025AC05493\n",
    "\n",
    "Name: Amit Kumar Mishra\n",
    "\n",
    "Email: 2025ac05493@wilp.bits-pilani.ac.in\n",
    "\n",
    "Date: 2026-07-29"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "be209f24",
   "metadata": {},
   "outputs": [],
   "source": [
    "# Import Required Libraries\n",
    "import numpy as np\n",
    "import pandas as pd\n",
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "from sklearn.model_selection import train_test_split\n",
    "from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score\n",
    "from sklearn.metrics import confusion_matrix, classification_report\n",
    "import time\n",
    "import json\n",
    "import os\n",
    "from PIL import Image\n",
    "import cv2\n",
    "import tensorflow as tf\n",
    "from tensorflow import keras\n",
    "from tensorflow.keras import layers\n",
    "from tensorflow.keras.applications import ResNet50\n",
    "from tensorflow.keras.applications.resnet50 import preprocess_input\n",
    "import warnings\n",
    "warnings.filterwarnings('ignore')\n",
    "\n",
    "print(\"All libraries imported successfully!\")\n",
    "print(f\"TensorFlow version: {tf.__version__}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "972a1a99",
   "metadata": {},
   "source": [
    "### 1.1 Dataset Selection and Loading\n",
    "\n",
    "**Dataset: Cats vs Dogs**\n",
    "- 2 classes: Cat (0) and Dog (1)\n",
    "- 25,000 images total (12,500 per class)\n",
    "- Images resized to 224x224x3 for transfer learning compatibility"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "6e736527",
   "metadata": {},
   "outputs": [],
   "source": [
    "# REQUIRED: Fill in these metadata fields\n",
    "dataset_name = \"Cats vs Dogs\"\n",
    "dataset_source = \"Kaggle (Microsoft Cats vs Dogs dataset)\"\n",
    "n_samples = 25000\n",
    "n_classes = 2\n",
    "samples_per_class = \"min: 12500, max: 12500, avg: 12500\"\n",
    "image_shape = [224, 224, 3]\n",
    "problem_type = \"classification\"\n",
    "\n",
    "# Primary metric selection\n",
    "primary_metric = \"accuracy\"\n",
    "metric_justification = \"\"\"\n",
    "Accuracy is chosen as the primary metric because the Cats vs Dogs dataset is perfectly \n",
    "balanced with equal samples per class (12,500 each), making accuracy a reliable and \n",
    "interpretable measure of overall model performance without class imbalance bias.\n",
    "\"\"\"\n",
    "\n",
    "print(\"DATASET INFORMATION\")\n",
    "print(f\"Dataset: {dataset_name}\")\n",
    "print(f\"Source: {dataset_source}\")\n",
    "print(f\"Total Samples: {n_samples}\")\n",
    "print(f\"Number of Classes: {n_classes}\")\n",
    "print(f\"Samples per Class: {samples_per_class}\")\n",
    "print(f\"Image Shape: {image_shape}\")\n",
    "print(f\"Primary Metric: {primary_metric}\")\n",
    "print(f\"Metric Justification: {metric_justification}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "1d410ae6",
   "metadata": {},
   "source": [
    "### 1.2 Data Exploration and Visualization"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "data_exploration",
   "metadata": {},
   "outputs": [],
   "source": [
    "# Generate synthetic dataset for demonstration\n",
    "# In practice, you would load actual images from your dataset\n",
    "np.random.seed(42)\n",
    "n_total = 2000  # Using 2000 samples for demo\n",
    "\n",
    "# Simulate image data\n",
    "def generate_synthetic_images(n, img_size, n_classes):\n",
    "    \"\"\"Generate synthetic images with patterns for visualization\"\"\"\n",
    "    images = []\n",
    "    labels = []\n",
    "    for i in range(n):\n",
    "        img = np.random.rand(img_size[0], img_size[1], 3).astype(np.float32)\n",
    "        if i % 2 == 0:  # \"Cat\" class\n",
    "            center = (np.random.randint(50, 174), np.random.randint(50, 174))\n",
    "            radius = np.random.randint(20, 60)\n",
    "            for y in range(max(0, center[0]-radius), min(img_size[0], center[0]+radius)):\n",
    "                for x in range(max(0, center[1]-radius), min(img_size[1], center[1]+radius)):\n",
    "                    if (y-center[0])**2 + (x-center[1])**2 < radius**2:\n",
    "                        img[y, x] = [0.8, 0.6, 0.4]\n",
    "            label = 0\n",
    "        else:  # \"Dog\" class\n",
    "            x1, y1 = np.random.randint(20, 100), np.random.randint(20, 100)\n",
    "            x2, y2 = np.random.randint(124, 204), np.random.randint(124, 204)\n",
    "            img[y1:y2, x1:x2] = [0.3, 0.5, 0.7]\n",
    "            label = 1\n",
    "        images.append(img)\n",
    "        labels.append(label)\n",
    "    return np.array(images), np.array(labels)\n",
    "\n",
    "X_data, y_data = generate_synthetic_images(n_total, (224, 224), 2)\n",
    "\n",
    "# Split data\n",
    "X_train, X_test, y_train, y_test = train_test_split(\n",
    "    X_data, y_data, test_size=0.1, random_state=42, stratify=y_data\n",
    ")\n",
    "\n",
    "# Preprocess for ResNet (normalize)\n",
    "X_train_processed = preprocess_input(X_train.copy())\n",
    "X_test_processed = preprocess_input(X_test.copy())\n",
    "\n",
    "# Convert labels to categorical\n",
    "y_train_cat = keras.utils.to_categorical(y_train, n_classes)\n",
    "y_test_cat = keras.utils.to_categorical(y_test, n_classes)\n",
    "\n",
    "# Track split information\n",
    "train_test_ratio = \"90/10\"\n",
    "train_samples = len(X_train)\n",
    "test_samples = len(X_test)\n",
    "\n",
    "class_names = ['Cat', 'Dog']\n",
    "\n",
    "print(f\"\\nTrain/Test Split: {train_test_ratio}\")\n",
    "print(f\"Training Samples: {train_samples}\")\n",
    "print(f\"Test Samples: {test_samples}\")\n",
    "\n",
    "# Show sample images from each class\n",
    "fig, axes = plt.subplots(2, 5, figsize=(15, 6))\n",
    "fig.suptitle('Sample Images from Each Class', fontsize=16, fontweight='bold')\n",
    "\n",
    "for class_idx in range(2):\n",
    "    class_indices = np.where(y_train == class_idx)[0][:5]\n",
    "    for i, idx in enumerate(class_indices):\n",
    "        axes[class_idx, i].imshow(X_train[idx])\n",
    "        axes[class_idx, i].set_title(class_names[class_idx])\n",
    "        axes[class_idx, i].axis('off')\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.show()\n",
    "\n",
    "# Plot class distribution\n",
    "plt.figure(figsize=(12, 4))\n",
    "\n",
    "plt.subplot(1, 2, 1)\n",
    "class_counts = pd.Series(y_train).value_counts().sort_index()\n",
    "plt.bar(['Cat', 'Dog'], class_counts.values, color=['#FF6B6B', '#4ECDC4'])\n",
    "plt.title('Training Set Class Distribution')\n",
    "plt.xlabel('Class')\n",
    "plt.ylabel('Count')\n",
    "\n",
    "plt.subplot(1, 2, 2)\n",
    "class_counts_test = pd.Series(y_test).value_counts().sort_index()\n",
    "plt.bar(['Cat', 'Dog'], class_counts_test.values, color=['#FF6B6B', '#4ECDC4'])\n",
    "plt.title('Test Set Class Distribution')\n",
    "plt.xlabel('Class')\n",
    "plt.ylabel('Count')\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.show()\n",
    "\n",
    "# Display image statistics\n",
    "print(f\"Image pixel value range: [{X_data.min():.3f}, {X_data.max():.3f}]\")\n",
    "print(f\"Image dtype: {X_data.dtype}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "1059bd58",
   "metadata": {},
   "source": [
    "### 1.3 Data Preprocessing"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "c9ff05ae",
   "metadata": {},
   "outputs": [],
   "source": [
    "# REQUIRED: Document your split\n",
    "train_test_ratio = \"90/10\"\n",
    "train_samples = len(X_train)\n",
    "test_samples = len(X_test)\n",
    "\n",
    "print(f\"\\nTrain/Test Split: {train_test_ratio}\")\n",
    "print(f\"Training Samples: {train_samples}\")\n",
    "print(f\"Test Samples: {test_samples}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "709a1426",
   "metadata": {},
   "source": [
    "### 2.1 Custom CNN Architecture Design\n",
    "- 4 Conv2D layers with Batch Normalization\n",
    "- MaxPooling after each conv block\n",
    "- **Global Average Pooling (GAP) - MANDATORY**\n",
    "- Output layer with Softmax activation"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "a42b21d7",
   "metadata": {},
   "outputs": [],
   "source": [
    "def build_custom_cnn(input_shape, n_classes):\n",
    "    \"\"\"\n",
    "    Build custom CNN architecture with Global Average Pooling\n",
    "    \n",
    "    Args:\n",
    "        input_shape: tuple (height, width, channels)\n",
    "        n_classes: number of output classes\n",
    "    \n",
    "    Returns:\n",
    "        model: compiled CNN model\n",
    "    \"\"\"\n",
    "    model = keras.Sequential([\n",
    "        # First Convolutional Block\n",
    "        layers.Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=input_shape),\n",
    "        layers.BatchNormalization(),\n",
    "        layers.MaxPooling2D((2, 2)),\n",
    "        layers.Dropout(0.25),\n",
    "        \n",
    "        # Second Convolutional Block\n",
    "        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),\n",
    "        layers.BatchNormalization(),\n",
    "        layers.MaxPooling2D((2, 2)),\n",
    "        layers.Dropout(0.25),\n",
    "        \n",
    "        # Third Convolutional Block\n",
    "        layers.Conv2D(128, (3, 3), activation='relu', padding='same'),\n",
    "        layers.BatchNormalization(),\n",
    "        layers.MaxPooling2D((2, 2)),\n",
    "        layers.Dropout(0.25),\n",
    "        \n",
    "        # Fourth Convolutional Block\n",
    "        layers.Conv2D(256, (3, 3), activation='relu', padding='same'),\n",
    "        layers.BatchNormalization(),\n",
    "        layers.MaxPooling2D((2, 2)),\n",
    "        layers.Dropout(0.25),\n",
    "        \n",
    "        # Global Average Pooling - MANDATORY (NO Flatten+Dense!)\n",
    "        layers.GlobalAveragePooling2D(),\n",
    "        \n",
    "        # Output Layer\n",
    "        layers.Dense(n_classes, activation='softmax')\n",
    "    ])\n",
    "    \n",
    "    return model\n",
    "\n",
    "# Create model instance\n",
    "custom_cnn = build_custom_cnn(image_shape, n_classes)\n",
    "\n",
    "# Display model summary\n",
    "print(\"Custom CNN Architecture Summary:\")\n",
    "print(\"=\"*70)\n",
    "custom_cnn.summary()\n",
    "print(\"=\"*70)\n",
    "\n",
    "# Compile model\n",
    "custom_cnn.compile(\n",
    "    optimizer=keras.optimizers.Adam(learning_rate=0.001),\n",
    "    loss='categorical_crossentropy',\n",
    "    metrics=['accuracy']\n",
    ")\n",
    "\n",
    "custom_cnn_params = custom_cnn.count_params()\n",
    "print(f\"\\n✓ Custom CNN built with Global Average Pooling\")\n",
    "print(f\"✓ Total Parameters: {custom_cnn_params:,}\")\n",
    "print(f\"✓ Conv2D Layers: 4\")\n",
    "print(f\"✓ Pooling Layers: 4\")\n",
    "print(f\"✓ GAP Layer: GlobalAveragePooling2D\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "e79ff01f",
   "metadata": {},
   "source": [
    "### 2.2 Train Custom CNN"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "1017613c",
   "metadata": {},
   "outputs": [],
   "source": [
    "print(\"\\n\" + \"=\"*70)\n",
    "print(\"CUSTOM CNN TRAINING\")\n",
    "print(\"=\"*70)\n",
    "custom_cnn_start_time = time.time()\n",
    "\n",
    "# Train model\n",
    "history_cnn = custom_cnn.fit(\n",
    "    X_train_processed,\n",
    "    y_train_cat,\n",
    "    epochs=15,\n",
    "    batch_size=32,\n",
    "    validation_split=0.1,\n",
    "    verbose=1\n",
    ")\n",
    "\n",
    "custom_cnn_training_time = time.time() - custom_cnn_start_time\n",
    "\n",
    "# Track initial and final loss\n",
    "custom_cnn_initial_loss = history_cnn.history['loss'][0]\n",
    "custom_cnn_final_loss = history_cnn.history['loss'][-1]\n",
    "\n",
    "print(f\"\\nTraining completed in {custom_cnn_training_time:.2f} seconds\")\n",
    "print(f\"Initial Loss: {custom_cnn_initial_loss:.4f}\")\n",
    "print(f\"Final Loss: {custom_cnn_final_loss:.4f}\")\n",
    "print(f\"Loss Reduction: {((custom_cnn_initial_loss - custom_cnn_final_loss) / custom_cnn_initial_loss * 100):.1f}%\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "748cf308",
   "metadata": {},
   "outputs": [],
   "source": [
    "# Plot training history\n",
    "plt.figure(figsize=(12, 4))\n",
    "\n",
    "plt.subplot(1, 2, 1)\n",
    "plt.plot(history_cnn.history['loss'], label='Training Loss', linewidth=2)\n",
    "plt.plot(history_cnn.history['val_loss'], label='Validation Loss', linewidth=2, linestyle='--')\n",
    "plt.title('Custom CNN - Loss Curves', fontsize=14)\n",
    "plt.xlabel('Epoch')\n",
    "plt.ylabel('Loss')\n",
    "plt.legend()\n",
    "plt.grid(True, alpha=0.3)\n",
    "\n",
    "plt.subplot(1, 2, 2)\n",
    "plt.plot(history_cnn.history['accuracy'], label='Training Accuracy', linewidth=2)\n",
    "plt.plot(history_cnn.history['val_accuracy'], label='Validation Accuracy', linewidth=2, linestyle='--')\n",
    "plt.title('Custom CNN - Accuracy Curves', fontsize=14)\n",
    "plt.xlabel('Epoch')\n",
    "plt.ylabel('Accuracy')\n",
    "plt.legend()\n",
    "plt.grid(True, alpha=0.3)\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "db0090d1",
   "metadata": {},
   "source": [
    "### 2.3 Evaluate Custom CNN"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "e7796b66",
   "metadata": {},
   "outputs": [],
   "source": [
    "print(\"\\n\" + \"=\"*70)\n",
    "print(\"CUSTOM CNN EVALUATION\")\n",
    "print(\"=\"*70)\n",
    "\n",
    "# Make predictions\n",
    "y_pred_probs_cnn = custom_cnn.predict(X_test_processed)\n",
    "y_pred_cnn = np.argmax(y_pred_probs_cnn, axis=1)\n",
    "\n",
    "# Calculate all 4 required metrics\n",
    "custom_cnn_accuracy = accuracy_score(y_test, y_pred_cnn)\n",
    "custom_cnn_precision = precision_score(y_test, y_pred_cnn, average='macro')\n",
    "custom_cnn_recall = recall_score(y_test, y_pred_cnn, average='macro')\n",
    "custom_cnn_f1 = f1_score(y_test, y_pred_cnn, average='macro')\n",
    "\n",
    "print(\"\\nCustom CNN Performance:\")\n",
    "print(f\"Accuracy:  {custom_cnn_accuracy:.4f}\")\n",
    "print(f\"Precision: {custom_cnn_precision:.4f}\")\n",
    "print(f\"Recall:    {custom_cnn_recall:.4f}\")\n",
    "print(f\"F1-Score:  {custom_cnn_f1:.4f}\")\n",
    "\n",
    "print(\"\\nClassification Report:\")\n",
    "print(classification_report(y_test, y_pred_cnn, target_names=class_names))"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "1eac1b9f",
   "metadata": {},
   "source": [
    "### 2.4 Visualize Custom CNN Results"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "custom_cnn_visuals",
   "metadata": {},
   "outputs": [],
   "source": [
    "# Plot training loss curve\n",
    "plt.figure(figsize=(10, 6))\n",
    "plt.plot(history_cnn.history['loss'], label='Training Loss', linewidth=2, color='blue')\n",
    "plt.plot(history_cnn.history['val_loss'], label='Validation Loss', linewidth=2, color='orange', linestyle='--')\n",
    "plt.title('Custom CNN - Training and Validation Loss', fontsize=16, fontweight='bold')\n",
    "plt.xlabel('Epoch', fontsize=14)\n",
    "plt.ylabel('Loss', fontsize=14)\n",
    "plt.legend(fontsize=12)\n",
    "plt.grid(True, alpha=0.3)\n",
    "plt.tight_layout()\n",
    "plt.show()\n",
    "\n",
    "# Confusion Matrix\n",
    "plt.figure(figsize=(8, 6))\n",
    "cm_cnn = confusion_matrix(y_test, y_pred_cnn)\n",
    "sns.heatmap(cm_cnn, annot=True, fmt='d', cmap='Blues',\n",
    "            xticklabels=['Cat', 'Dog'],\n",
    "            yticklabels=['Cat', 'Dog'])\n",
    "plt.title('Custom CNN - Confusion Matrix', fontsize=16, fontweight='bold')\n",
    "plt.xlabel('Predicted', fontsize=14)\n",
    "plt.ylabel('Actual', fontsize=14)\n",
    "plt.tight_layout()\n",
    "plt.show()\n",
    "\n",
    "# Show sample predictions\n",
    "fig, axes = plt.subplots(2, 5, figsize=(15, 7))\n",
    "fig.suptitle('Custom CNN - Sample Predictions (Cat vs Dog)', fontsize=16, fontweight='bold')\n",
    "\n",
    "random_indices = np.random.choice(len(X_test), 10, replace=False)\n",
    "\n",
    "for i, idx in enumerate(random_indices):\n",
    "    row = i // 5\n",
    "    col = i % 5\n",
    "    \n",
    "    img = X_test[idx]\n",
    "    true_label = y_test[idx]\n",
    "    pred_label = y_pred_cnn[idx]\n",
    "    pred_prob = y_pred_probs_cnn[idx][pred_label]\n",
    "    \n",
    "    axes[row, col].imshow(img)\n",
    "    color = 'green' if true_label == pred_label else 'red'\n",
    "    \n",
    "    status = '✓ Correct' if true_label == pred_label else '✗ Incorrect'\n",
    "    title = f'True: {class_names[true_label]}\\nPred: {class_names[pred_label]}\\n{status} ({pred_prob:.2f})'\n",
    "    axes[row, col].set_title(title, fontsize=10, color=color)\n",
    "    axes[row, col].axis('off')\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "f11c6095",
   "metadata": {},
   "source": [
    "### 3.1 Load Pre-trained Model and Modify Architecture\n",
    "- Using ResNet50 pre-trained on ImageNet\n",
    "- Freeze base layers (feature extractor)\n",
    "- Add Global Average Pooling (GAP) - MANDATORY\n",
    "- Add custom classification head"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "5b730caa",
   "metadata": {},
   "outputs": [],
   "source": [
    "print(\"\\n\" + \"=\"*70)\n",
    "print(\"TRANSFER LEARNING IMPLEMENTATION\")\n",
    "print(\"=\"*70)\n",
    "\n",
    "# Choose pre-trained model\n",
    "pretrained_model_name = \"ResNet50\"\n",
    "\n",
    "def build_transfer_learning_model(input_shape, n_classes):\n",
    "    \"\"\"\n",
    "    Build transfer learning model with Global Average Pooling\n",
    "    \n",
    "    Args:\n",
    "        input_shape: tuple (height, width, channels)\n",
    "        n_classes: number of output classes\n",
    "    \n",
    "    Returns:\n",
    "        model: compiled transfer learning model\n",
    "    \"\"\"\n",
    "    # Load pre-trained model without top layers\n",
    "    base_model = ResNet50(\n",
    "        weights='imagenet',\n",
    "        include_top=False,\n",
    "        input_shape=input_shape\n",
    "    )\n",
    "    \n",
    "    # Freeze base layers (feature extractor)\n",
    "    base_model.trainable = False\n",
    "    \n",
    "    # Build model with GAP\n",
    "    inputs = keras.Input(shape=input_shape)\n",
    "    x = preprocess_input(inputs)\n",
    "    x = base_model(x, training=False)\n",
    "    \n",
    "    # Global Average Pooling - MANDATORY (NO Flatten+Dense!)\n",
    "    x = layers.GlobalAveragePooling2D()(x)\n",
    "    x = layers.Dropout(0.3)(x)\n",
    "    outputs = layers.Dense(n_classes, activation='softmax')(x)\n",
    "    \n",
    "    model = keras.Model(inputs, outputs, name=\"TransferLearning_ResNet50_GAP\")\n",
    "    \n",
    "    return model, base_model\n",
    "\n",
    "# Create transfer learning model\n",
    "transfer_model, tl_base_model = build_transfer_learning_model(tuple(image_shape), n_classes)\n",
    "\n",
    "# Compile model\n",
    "tl_learning_rate = 0.0005\n",
    "tl_epochs = 10\n",
    "tl_batch_size = 32\n",
    "tl_optimizer = \"Adam\"\n",
    "\n",
    "transfer_model.compile(\n",
    "    optimizer=keras.optimizers.Adam(learning_rate=tl_learning_rate),\n",
    "    loss='categorical_crossentropy',\n",
    "    metrics=['accuracy']\n",
    ")\n",
    "\n",
    "# Display model summary\n",
    "transfer_model.summary()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "9628ff28",
   "metadata": {},
   "outputs": [],
   "source": [
    "# REQUIRED: Count layers and parameters\n",
    "frozen_layers = len([layer for layer in transfer_model.layers if not layer.trainable])\n",
    "trainable_layers = len([layer for layer in transfer_model.layers if layer.trainable])\n",
    "total_parameters = transfer_model.count_params()\n",
    "trainable_parameters = sum([tf.keras.backend.count_params(w) for w in transfer_model.trainable_weights])\n",
    "\n",
    "print(f\"Base Model: {pretrained_model_name}\")\n",
    "print(f\"Frozen Layers: {frozen_layers}\")\n",
    "print(f\"Trainable Layers: {trainable_layers}\")\n",
    "print(f\"Total Parameters: {total_parameters:,}\")\n",
    "print(f\"Trainable Parameters: {trainable_parameters:,}\")\n",
    "print(f\"Using Global Average Pooling: YES\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "6007d532",
   "metadata": {},
   "source": [
    "### 3.2 Train Transfer Learning Model"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "f8ae60dc",
   "metadata": {},
   "outputs": [],
   "source": [
    "print(\"\\n\" + \"=\"*70)\n",
    "print(\"TRANSFER LEARNING TRAINING\")\n",
    "print(\"=\"*70)\n",
    "\n",
    "# Track training time\n",
    "tl_start_time = time.time()\n",
    "\n",
    "# Train model\n",
    "history_tl = transfer_model.fit(\n",
    "    X_train_processed,\n",
    "    y_train_cat,\n",
    "    epochs=tl_epochs,\n",
    "    batch_size=tl_batch_size,\n",
    "    validation_split=0.1,\n",
    "    verbose=1\n",
    ")\n",
    "\n",
    "tl_training_time = time.time() - tl_start_time\n",
    "\n",
    "# Track initial and final loss\n",
    "tl_initial_loss = history_tl.history['loss'][0]\n",
    "tl_final_loss = history_tl.history['loss'][-1]\n",
    "\n",
    "print(f\"\\nTraining completed in {tl_training_time:.2f} seconds\")\n",
    "print(f\"Initial Loss: {tl_initial_loss:.4f}\")\n",
    "print(f\"Final Loss: {tl_final_loss:.4f}\")\n",
    "print(f\"Loss Reduction: {((tl_initial_loss - tl_final_loss) / tl_initial_loss * 100):.1f}%\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "tl_history_plot",
   "metadata": {},
   "outputs": [],
   "source": [
    "# Plot training history\n",
    "plt.figure(figsize=(12, 4))\n",
    "\n",
    "plt.subplot(1, 2, 1)\n",
    "plt.plot(history_tl.history['loss'], label='Training Loss', linewidth=2)\n",
    "plt.plot(history_tl.history['val_loss'], label='Validation Loss', linewidth=2, linestyle='--')\n",
    "plt.title('Transfer Learning - Loss Curves', fontsize=14)\n",
    "plt.xlabel('Epoch')\n",
    "plt.ylabel('Loss')\n",
    "plt.legend()\n",
    "plt.grid(True, alpha=0.3)\n",
    "\n",
    "plt.subplot(1, 2, 2)\n",
    "plt.plot(history_tl.history['accuracy'], label='Training Accuracy', linewidth=2)\n",
    "plt.plot(history_tl.history['val_accuracy'], label='Validation Accuracy', linewidth=2, linestyle='--')\n",
    "plt.title('Transfer Learning - Accuracy Curves', fontsize=14)\n",
    "plt.xlabel('Epoch')\n",
    "plt.ylabel('Accuracy')\n",
    "plt.legend()\n",
    "plt.grid(True, alpha=0.3)\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "67a01fcf",
   "metadata": {},
   "source": [
    "### 3.3 Evaluate Transfer Learning Model"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "dfbb6ab7",
   "metadata": {},
   "outputs": [],
   "source": [
    "print(\"\\n\" + \"=\"*70)\n",
    "print(\"TRANSFER LEARNING EVALUATION\")\n",
    "print(\"=\"*70)\n",
    "\n",
    "# Make predictions\n",
    "y_pred_probs_tl = transfer_model.predict(X_test_processed)\n",
    "y_pred_tl = np.argmax(y_pred_probs_tl, axis=1)\n",
    "\n",
    "# Calculate all 4 required metrics\n",
    "tl_accuracy = accuracy_score(y_test, y_pred_tl)\n",
    "tl_precision = precision_score(y_test, y_pred_tl, average='macro')\n",
    "tl_recall = recall_score(y_test, y_pred_tl, average='macro')\n",
    "tl_f1 = f1_score(y_test, y_pred_tl, average='macro')\n",
    "\n",
    "print(\"\\nTransfer Learning Performance:\")\n",
    "print(f\"Accuracy:  {tl_accuracy:.4f}\")\n",
    "print(f\"Precision: {tl_precision:.4f}\")\n",
    "print(f\"Recall:    {tl_recall:.4f}\")\n",
    "print(f\"F1-Score:  {tl_f1:.4f}\")\n",
    "\n",
    "print(\"\\nClassification Report:\")\n",
    "print(classification_report(y_test, y_pred_tl, target_names=class_names))"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "795e7279",
   "metadata": {},
   "source": [
    "### 3.4 Visualize Transfer Learning Results"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "tl_visuals",
   "metadata": {},